# CE541E08 — Unit 2 · Day 14 — enumerate, zip, sorted and Weibull Flood Frequency
| | |
|---|---|
| **Course** | CE541E08 |
| **Department** | Civil Engineering · Christ University |
| **Instructor** | Dr. Arpan Pradhan |
| **Unit** | Unit 2 |
| **Session** | Day 14 of 45 |
| **CO** | CO2 |
| **Topics** | enumerate · zip · sorted() · AMS Weibull · water balance · return period |
---
> Read the explanation before each code block. Check the expected output. Run the cell and verify. Then try the small challenge.
---

In [ ]:
student_name = "Your Full Name"
roll_number  = "2024XXXXXX"
github_repo  = "https://github.com/your-username/CE541E08-2026"
session      = "Day 14"
print(f"CE541E08 | {student_name} | {roll_number} | {session}")

---
## Section 1 — Pairing Data with zip and enumerate

Often in hydrology, multiple datasets are collected simultaneously — daily streamflow alongside daily rainfall, station names alongside measurements. Python's `zip()` pairs corresponding elements from two or more lists, allowing simultaneous iteration.

---
## Code Block 1 — enumerate: Flood Days Report

### What this code does

We process a 14-day streamflow record using `enumerate`, identifying flood days (above a threshold), computing the mean, and printing a report with day numbers.

### Why each step is taken

**`mean_flow = sum(daily_flow_cumecs) / len(daily_flow_cumecs)`:**
Computing the mean before the loop allows the if-check `if Q >= flood_threshold` to use this value.

**`for day, Q in enumerate(daily_flow_cumecs, 1)`:**
Day numbers start at 1 (not 0). The second argument to `enumerate` sets the starting count.

**`if Q >= flood_threshold`:**
The `flood_threshold` is the mean — any day above the mean is counted as a high-flow day for this analysis.

### Expected output

```
Day    Flow (m3/s)   Mean    Flag
────────────────────────────────────────
  1       234.5    534.5    —
  2       267.8    534.5    —
  3       312.4    534.5    —
  4       890.2    534.5   FLOOD
  5      1245.6    534.5   FLOOD
...
Flood days: 5 of 14  Mean: 534.5 m3/s
```

In [ ]:
daily_flow_cumecs = [
    234.5, 267.8, 312.4, 890.2, 1245.6, 987.3, 756.4,
    543.2, 412.8, 345.6, 289.4, 245.1, 212.3, 198.7
]

mean_flow       = sum(daily_flow_cumecs) / len(daily_flow_cumecs)
flood_threshold = mean_flow     # above mean = "flood day" for this study
flood_count     = 0

print(f"{'Day':>4} {'Flow (m3/s)':>14} {'Mean':>8} {'Flag'}")
print("─"*44)

for day, Q in enumerate(daily_flow_cumecs, 1):
    flag = ""
    if Q >= flood_threshold:
        flag = "FLOOD"
        flood_count += 1
    print(f"{day:>4} {Q:>14.1f} {mean_flow:>8.1f} {flag}")

print(f"
Flood days: {flood_count} of {len(daily_flow_cumecs)}  Mean: {mean_flow:.1f} m3/s")

### 🔁 Try this

Change the flood threshold to a fixed value of 600 m³/s (a physical flood stage).

How many days are now classified as flood days? Is this more or less than using the mean?

---
## Code Block 2 — zip: Rainfall-Runoff Coefficients

### What this code does

We use `zip()` to pair daily rainfall and runoff observations, compute the runoff coefficient for each rainy day, and print a comparison table.

### Why each step is taken

**`zip(rainfall_mm, runoff_mm, dates)`:**
`zip` combines three lists into triples — on each iteration, you get (rain, runoff, date) for one day. The loop stops when the shortest list is exhausted.

**`if rain > 0:`:**
The runoff coefficient C = runoff/rainfall is only meaningful when it rained. On dry days, dividing by zero would raise an error.

**`C = runoff/rain`:**
The rational method coefficient: what fraction of rainfall became runoff. Values typically range from 0 (all infiltration) to 1 (all runoff).

### Expected output

```
Date       Rainfall  Runoff     C
─────────────────────────────────────
Jul-01        0.0     0.0     —
Jul-02       23.4     8.2    0.350
Jul-03       67.8    28.4    0.419
Jul-04      134.5    72.1    0.536
Jul-05       89.2    41.3    0.463
Jul-06       45.6    18.7    0.410
Jul-07       12.3     2.8    0.228

Mean C (rainy days): 0.401
```

In [ ]:
rainfall_mm = [0.0, 23.4, 67.8, 134.5, 89.2, 45.6, 12.3]
runoff_mm   = [0.0,  8.2, 28.4,  72.1, 41.3, 18.7,  2.8]
dates       = ["Jul-01","Jul-02","Jul-03","Jul-04","Jul-05","Jul-06","Jul-07"]

print(f"{'Date':<10} {'Rainfall':>8} {'Runoff':>7} {'C':>6}")
print("─"*37)

C_list = []
for date, rain, runoff in zip(dates, rainfall_mm, runoff_mm):
    if rain > 0:
        C = runoff / rain
        C_list.append(C)
        print(f"{date:<10} {rain:>8.1f} {runoff:>7.1f} {C:>6.3f}")
    else:
        print(f"{date:<10} {rain:>8.1f} {runoff:>7.1f} {'—':>6}")

mean_C = sum(C_list) / len(C_list)
print(f"
Mean C (rainy days): {mean_C:.3f}")

### 🔁 Try this

Compute the weighted mean C (weighted by rainfall depth):

`weighted_C = sum(r*c for r,c in zip(rainfall_mm, C_list_with_zeros)) / sum(rainfall_mm)`

Which is higher — the simple mean or the rainfall-weighted mean? Why might heavy-rain days have higher C?

---
## Code Block 3 — sorted and Weibull Flood Frequency Analysis

### What this code does

We perform a complete Weibull flood frequency analysis on a 20-year Annual Maximum Series — computing return periods and estimating the 10-year, 25-year, and 50-year design floods.

### Why each step is taken

**`sorted(AMS, reverse=True)`:**
`sorted()` returns a new sorted list without modifying the original. `reverse=True` gives descending order — largest flood gets rank 1.

**`AMS.index(peak) + 1`:**
`.index(value)` returns the 0-based position of a value. Adding 1 converts to a 1-based year number.

**Weibull plotting position `P = rank/(n+1)`:**
The Weibull formula avoids P=0 and P=1. For rank 1 of 20 events: P = 1/21 = 0.0476, T = 21 years.

**Linear interpolation for T-year floods:**
We find the two ranked flows that bracket the target exceedance probability (1/T) and linearly interpolate between them.

### Algorithm

```
1. AMS = 20-year peak flow list
   sorted descending → AMS_sorted
   n = 20

2. Weibull:
   for rank, flow in enumerate(AMS_sorted, 1):
     P = rank/(n+1)
     T = 1/P
     print rank, flow, P, T

3. Linear interpolation for T=10,25,50:
   find P_target = 1/T
   find bracket in P_list
   interpolate flow
```

### Expected output

```
AMS statistics: mean=892.9, peak=2134.6, min=345.2 m3/s
 Rank   Flow   P(exceed)   T(years)
   1  2134.6    0.0476      21.00
   2  1890.2    0.0952      10.50
   ...
  20   345.2    0.9524       1.05
T=10-yr flood: ~1800 m3/s
```

In [ ]:
AMS = [
    345.2, 567.8, 892.3, 1234.5, 456.7, 789.0, 1567.8,
    678.9, 345.6, 1890.2, 567.3, 789.4, 2134.6, 456.8,
    678.9, 1234.5, 345.7, 890.2, 1456.7, 567.4
]

n       = len(AMS)
total   = sum(AMS)
mean    = total / n
peak    = max(AMS)
minimum = min(AMS)
peak_yr = AMS.index(peak) + 1
min_yr  = AMS.index(minimum) + 1

print(f"AMS statistics: mean={mean:.1f}, peak={peak}, min={minimum} m3/s")
print()

AMS_sorted = sorted(AMS, reverse=True)
P_list     = []
Q_list     = []

print(f"{'Rank':>5} {'Flow':>8} {'P(exceed)':>11} {'T(years)':>10}")
print("─"*38)
for rank, flow in enumerate(AMS_sorted, 1):
    P = rank / (n + 1)
    T = 1 / P
    P_list.append(P); Q_list.append(flow)
    print(f"{rank:>5} {flow:>8.1f} {P:>11.4f} {T:>10.2f}")

# Simple linear interpolation for T-year floods
for T_target in [10, 25, 50]:
    P_target = 1 / T_target
    for j in range(len(P_list)-1):
        if P_list[j] <= P_target <= P_list[j+1]:
            frac = (P_target - P_list[j]) / (P_list[j+1] - P_list[j])
            Q_T  = Q_list[j] + frac * (Q_list[j+1] - Q_list[j])
            print(f"T={T_target:>3}-yr flood: {Q_T:.0f} m3/s")
            break

### 🔁 Try this

Add the T=2-year flood (median flood) to the interpolation loop.

The T=2-year flood corresponds to P=0.5 (exceeded 50% of the time). Is it close to the median of the AMS? Use `sorted(AMS)[len(AMS)//2]` to check.

---
## Session Summary — enumerate, zip, sorted

| Function | What it does | Example |
|---|---|---|
| `enumerate(lst, 1)` | Pairs (index, value) with custom start | Day numbers |
| `zip(a, b, c)` | Combines multiple lists element-wise | (date, rain, runoff) |
| `sorted(lst)` | Returns sorted copy (ascending) | `sorted(AMS)` |
| `sorted(lst, reverse=True)` | Descending sort | AMS rank table |
| `lst.index(value)` | 0-based position of value | `AMS.index(peak)` |
| Weibull P | `rank/(n+1)` | Exceedance probability |
| Return period T | `1/P` | Years between events |

---
## Day 14 Assignment

Full Weibull table for the given AMS. Estimate the T=2, 5, 10, 25-year design floods.

### ▶ Assignment cell

In [ ]:
AMS = [345.2,567.8,892.3,1234.5,456.7,789.0,1567.8,
       678.9,345.6,1890.2,567.3,789.4,2134.6,456.8,
       678.9,1234.5,345.7,890.2,1456.7,567.4]
n = len(AMS)

# Sort descending and compute return periods
AMS_sorted = sorted(AMS, reverse=True)
ranks = list(range(1, n+1))
probs = [r/(n+1) for r in ranks]
return_periods = [1/p for p in probs]

print(f"{'Rank':>5} {'Year':>6} {'Flow':>12} {'T (yr)':>10}")
print("-"*36)
for rank, flow, T in zip(ranks, AMS_sorted, return_periods):
    print(f"{rank:>5} {rank:>6} {flow:>12.1f} {T:>10.2f}")

---
- [ ] Run all cells — verify outputs match expected outputs
- [ ] Complete the assignment cell
- [ ] Upload: `Unit2_LoopsDecisions/CE541E08_U2_Day14.ipynb`
- [ ] Commit: `Day 14 assignment completed`

*CE541E08 · Civil Engineering · Christ University · 2026-27 · Dr. Arpan Pradhan*